# LC 371 — Sum of Two Integers
**Difficulty:** Medium | **Category:** Bit Manipulation
**Pattern:** XOR + Carry Simulation

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> XOR gives the sum without carry.
AND shifted left gives the carry. Repeat until carry is zero.
Python needs a 32-bit mask to prevent infinite loops from
its arbitrary-precision integers.
</div>

## Official Problem Statement

Given two integers `a` and `b`, return the sum of the two integers
without using the operators `+` and `-`.

**Constraints:**
- `-1000 <= a, b <= 1000`

## What This Is Actually Asking

Add two numbers using only bit operations — no + or - allowed.

This is how hardware adder circuits work at the silicon level.
XOR handles bits that don't produce a carry; AND finds where
a carry is needed; shift moves the carry to the next column.

Python's "infinite" integers require masking to 32 bits to
prevent the carry loop from running forever on negative numbers.

## Walk Through an Example by Hand

Input: `a = 3, b = 5`  →  expected: `8`

```
a = 011  (3)
b = 101  (5)

Iteration 1:
  sum   = a ^ b = 011 ^ 101 = 110  (6) ← bits without carry
  carry = (a & b) << 1
        = (011 & 101) << 1
        = 001 << 1 = 010  (2)           ← carry bits

Iteration 2: a=110, b=010
  sum   = 110 ^ 010 = 100  (4)
  carry = (110 & 010) << 1
        = 010 << 1 = 100  (4)

Iteration 3: a=100, b=100
  sum   = 100 ^ 100 = 000  (0)
  carry = (100 & 100) << 1
        = 100 << 1 = 1000  (8)

Iteration 4: a=0000, b=1000
  sum   = 0000 ^ 1000 = 1000  (8)
  carry = (0000 & 1000) << 1 = 0

carry == 0, return sum = 8  ✓
```

## The Picture

```
How a hardware half-adder works:

  Inputs:  A bit  B bit
  ─────────────────────────────────────────
  Sum bit:    A XOR B    (different → 1)
  Carry bit:  A AND B    (both 1 → carry)

Full addition = repeat half-adder until no carry:

  a = 0011  (3)
  b = 0101  (5)
  ─────────────────────
  XOR  = 0110  (no-carry sum)
  AND  = 0001  (where carries occur)
  <<1  = 0010  (shift carry to next column)
  ─────────────────────
  new a = 0110, new b = 0010  → repeat

Python 32-bit mask: 0xFFFFFFFF
  Keeps numbers in 32-bit range.
  After loop, convert negative two's-complement back:
    if result > 0x7FFFFFFF: result -= 0x100000000
```

## When To Use This Pattern

- When asked to add (or subtract) **without arithmetic operators**.
- When simulating **hardware-level arithmetic** in software.
- When you see the phrase "bit manipulation only" in a problem.
- When implementing a custom integer type
  that needs add/subtract without using the CPU adder directly.

## The Approach

Apply a 32-bit mask (0xFFFFFFFF) throughout to keep Python in
fixed-width integer behavior. While b (carry) is not zero,
compute the new sum as (a ^ b) & MASK and the new carry as
((a & b) << 1) & MASK. After the loop, if the result fits
in a signed 32-bit int (> 0x7FFFFFFF), subtract 0x100000000
to recover the correct negative value.

In [1]:
from typing import List  # standard collection types

In [2]:
def test_harness(func):
    tests = [
        # (a, b, expected, label)
        (1,   2,    3, "basic: 1+2"),
        (3,   5,    8, "standard: 3+5"),
        (0,   0,    0, "edge: 0+0"),
        (-1,  1,    0, "negative + positive = 0"),
        (-3, -5,   -8, "both negative"),
        (1000, 1000, 2000, "max input"),
    ]
    passed = 0
    for a, b, expected, label in tests:
        result = func(a, b)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"  [{status}] {label}")
        if status == "FAILED":
            print(f"           got={result}, expected={expected}")
    print(f"\n  {passed}/{len(tests)} tests passed")

In [6]:
def getSum(a: int, b: int) -> int:
    """
    Return a + b without using + or - operators.

    Approach: XOR + carry simulation.
      XOR = sum without carry.
      AND << 1 = carry for next column.
      Repeat until carry is zero.
      Python mask 0xFFFFFFFF enforces 32-bit behavior.

    Time:  O(1) — at most 32 iterations
    Space: O(1)
    """
def getSum(a: int, b: int) -> int:
    MASK = 0xFFFFFFFF   # 32-bit fence — chops off any bits above position 31
    MAX  = 0x7FFFFFFF   # highest positive 32-bit int (0111...1) — sign bit is 0

    while b & MASK:                       # keep going while carry bits exist
        tmp = ((a & b) << 1) & MASK       # AND finds carry positions, shift moves them left, mask keeps it 32-bit
        a = (a ^ b) & MASK                # XOR adds without carry, mask keeps it 32-bit
        b = tmp                           # carry becomes the new b — next round absorbs it

    # Python has no 32-bit ceiling — negative results leaked above bit 31
    # if a > MAX, the sign bit (bit 31) is set — it's negative in 32-bit world
    # ~(a ^ MASK) flips it back to Python's signed negative representation
    return a if a <= MAX else ~(a ^ MASK)
'''r
3
8
0
-8
2000

'''

# Debug prints — expected values shown in comments
print(getSum(1, 2))       # expected: 3
print(getSum(3, 5))       # expected: 8
print(getSum(-1, 1))      # expected: 0
print(getSum(-3, -5))     # expected: -8
print(getSum(1000, 1000)) # expected: 2000

3
8
0
-8
2000


In [ ]:
# Uncomment and run when solution is ready
# test_harness(getSum)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Recursive XOR + carry | O(1) — max 32 iterations | O(1) |
| Iterative XOR + carry | O(1) — max 32 iterations | O(1) |

## Real World Connection

This is literally how CPUs add numbers in hardware ALUs.
In Citi's AWS data platform, understanding bitwise arithmetic
matters when writing low-level Kinesis or DynamoDB serializers
that pack numeric fields into fixed-width byte buffers.
Serverless Lambda functions processing high-throughput
telemetry from 6,000 endpoints benefit from bit-packing
aggregations that avoid costly integer arithmetic at scale.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra